In [1]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import polars as pl
import yaml
import spacy
from corextopic import corextopic as ct
from dvclive import Live
from matplotlib.figure import Figure
from spacy.tokens import DocBin

from job_post_nlp.utils.interactive import try_inter

try_inter()
from job_post_nlp.prepare import corpus_unpack, register_extensions, load_data,register_extensions, load_texts # noqa: E402
from job_post_nlp.utils.find_project_root import find_project_root  # noqa: E402
from job_post_nlp.evaluate import load_model  # noqa: E402
from job_post_nlp.train import load_corpus_split, load_tdm  # noqa: E402
import helpfuncs as hf

/home/b281467@PROD.SITAD.DK/.conda/envs/jobpostnlp/lib/python3.12/site-packages/requests/__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [2]:
# store in one dictionary
data = hf.load_everything()
model = data['model']

In [3]:
model.get_top_docs(topic=0, n_docs=10)

[('3939415', np.float64(0.0)),
 ('5548167', np.float64(0.0)),
 ('5548162', np.float64(0.0)),
 ('5548154', np.float64(0.0)),
 ('5548153', np.float64(0.0)),
 ('5548150', np.float64(0.0)),
 ('5548149', np.float64(0.0)),
 ('5548205', np.float64(0.0)),
 ('5548197', np.float64(0.0)),
 ('5548196', np.float64(0.0))]

In [4]:
hf.print_words_in_doc('2837330', data)

god, arbejde, positiv, søge, person, fremstille, best, western
hotel, jens, baggesen, kok, dygtig, alsidig, kreativ, overblik
selvstændig, glad, værdsætte, anden, mening, spændende, velsmagende, mad
selskab, konferencegæste, omhyggelig, køkkenhygiejne, periodevis, travl, periode, varebestilling
sammensætning, menue, sætter, pris, stabilie, kollegaere, god overblik, arbejde selvstændig


In [5]:
hf.print_words_and_text('2837330', data)

Text for document 2837330:
Best Western Hotel Jens Baggesen søger kok *Du er dygtig, alsidig og kreativ kom med godt
 overblik *Kan arbejde selvstændig *En glad og positiv person, der også værdsætter
 andres mening *Arbejde på mindre hotel *Fremstille spændende og velsmagende mad
 til selskaber og konferencegæster *Er omhyggelig med køkkenhygiejne *Periodevis
 travle perioder *Varebestilling og sammensætning af menuer *Sætter pris på gode
 og stabilie kollegaere
Words in document:
god, arbejde, positiv, søge, person, fremstille, best, western
hotel, jens, baggesen, kok, dygtig, alsidig, kreativ, overblik
selvstændig, glad, værdsætte, anden, mening, spændende, velsmagende, mad
selskab, konferencegæste, omhyggelig, køkkenhygiejne, periodevis, travl, periode, varebestilling
sammensætning, menue, sætter, pris, stabilie, kollegaere, god overblik, arbejde selvstændig


In [6]:
hf.print_random(data)


Vacancy ID: 5653069
Text for document 5653069:
Hedelyskolen er en folkeskole med omkring 730 elever, hvoraf de ca. 60 er tilknyttet
 en specialklasserække. Vi har også kommunens ordblindecenter og en modtageklasse
 for mellemtrinnet. Der er ca. 60 lærere på skolen og 15 medarbejdere i skolens SFO.
 Skolen ligger omgivet af grønne arealer med en stor legeplads og gode muligheder for
 udeskole. Skolens tre grundlæggende værdier er: Faglighed, Fællesskab og Livsduelighed.
 Vi lægger derfor vægt på at skabe gode og trygge relationer i et gensidigt
 forpligtende fællesskab, hvor fokus er på det hele menneske og den alsidige udvikling
 hos børnene. Vi kan tilbyde En skole med gode rammer for fysisk aktivitet og udendørs
 læringsmiljøer. Et stærkt fagligt miljø med et fælles fokus på børnenes trivsel,
 dannelse og læring. Et tæt samarbejde på årgangen samt med skolens SFO-pædagoger.
 En god stemning i personalegruppen og en åben og dialogsøgende ledelse. Vi forventer,
 at du Er uddannet pæda

åben, barn, skabe, miljø, ramme, fokus, ansættelse, børneattest
relevant, bidrage, stærkt, opmærksom, stemning, uddanne, engagere, information
afdelingsleder, ansætte, grøn, torsdag, fagligt, tæt, faglighed, henhold
gælde, ledelse, fælles, fællesskab, tirsdag, august, trivsel, gensidig
initiativrig, pædagog, pædagogisk, relation, personalegruppe, legeplads, børnenes, træffes
vilkår, indsats, skole, lærer, skol, areal, tre, trygg
indhentet, tilknytte, morten, udendørs, folkeskole, grundlæggende, referenc, mellemtrinn
årgang, sfo, didaktisk, hvoraf, læring, reflekterende, eventuel, ansættelsessamta
forpligte, aktivite, christiansen, udtalelse, børnehaveklasse, læringsmiljøer, udeskole, skolens
specialklasserække, helhedsorienteret, samarbejdskultur, dannelse, modtageklasse, angivn, livsduelighed, omgiv
dialogsøgende, hedelyskolen, ordblindecenter, lægge vægt, arbejde selvstændigt, god mulighed, gerne erfaring, tæt samarbejde
skabe god, del team, uddanne pædagog, forvente uddanne, god ram

In [7]:
ids_sorted = model.word_freq.argsort()
np.array(model.words)[ids_sorted[-10:]]

array(['ansøgning', 'opgave', 'tilbyde', 'samarbejde', 'samt', 'erfaring',
       'stilling', 'god', 'arbejde', 'søge'], dtype='<U100')